In [9]:
import os
import re
from collections import defaultdict
import cv2
import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [10]:
# 1. Define Paths and Configuration
BASE_DIR = r'C:\Users\hemanth\Desktop\3D_CNN\3D_CNN'
CLASSES = {'Control': 0, 'Diseased': 1}
IMG_SIZE = (64, 64)
TARGET_SLICES_PER_SUBJECT = 10

In [11]:
# 2. Extract Subject ID function
def extract_subject_id(filename):
  match = re.match(r'^(.*?)_\d+\.png$', filename, re.IGNORECASE)
  if match:
    return match.group(1)
  return filename.split('.')[0]

In [12]:
# 3. Load and Aggregate Dataset function
def load_and_aggregate_dataset(
    base_dir, classes, img_size, target_slices=10
):
  subject_data = defaultdict(list)
  subject_labels = {}

  for class_name, class_label in classes.items():
    class_folder = os.path.join(base_dir, class_name)
    if not os.path.exists(class_folder):
      print(f'Warning: Folder not found -> {class_folder}')
      continue

    for fname in os.listdir(class_folder):
      if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        subj_id = extract_subject_id(fname)
        img_path = os.path.join(class_folder, fname)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
          continue
        img_resized = cv2.resize(img, img_size)

        subject_data[subj_id].append(img_resized.flatten())
        subject_labels[subj_id] = class_label

  X_subjects = []
  y_subjects = []
  subject_ids = []

  for subj_id, slices in subject_data.items():
    if len(slices) >= target_slices:
      selected_slices = slices[:target_slices]
    else:
      selected_slices = slices + [
          np.zeros(img_size[0] * img_size[1], dtype=np.float32)
      ] * (target_slices - len(slices))

    concatenated_vector = np.concatenate(selected_slices, axis=0)
    X_subjects.append(concatenated_vector)
    y_subjects.append(subject_labels[subj_id])
    subject_ids.append(subj_id)

  return (
      np.array(X_subjects, dtype=np.float32),
      np.array(y_subjects),
      subject_ids,
  )

In [13]:
# 4. Execute Loading
print('Loading dataset with fixed slice shaping...')
X, y, subj_ids = load_and_aggregate_dataset(
    BASE_DIR, CLASSES, IMG_SIZE, TARGET_SLICES_PER_SUBJECT
)
print(f'Total Subjects Loaded: {len(X)}')
print(f'Feature vector length per subject: {X.shape[1]}')
print(
    f'Class Distribution -> Control (0): {np.sum(y == 0)}, Diseased (1):'
    f' {np.sum(y == 1)}'
)

Loading dataset with fixed slice shaping...
Total Subjects Loaded: 412
Feature vector length per subject: 40960
Class Distribution -> Control (0): 150, Diseased (1): 262


In [15]:
# 5. Evaluation Function for Baselines
def evaluate_baseline_model(model_name, classifier, X, y, n_splits=5):
  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  metrics = {
      'acc': [],
      'auc': [],
      'sens': [],
      'spec': [],
      'prec': [],
      'f1': [],
      'n_components': [],
  }

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    pca = PCA(n_components=0.95, random_state=42)

    X_train_scaled = scaler.fit_transform(X_train)
    X_train_pca = pca.fit_transform(X_train_scaled)

    X_val_scaled = scaler.transform(X_val)
    X_val_pca = pca.transform(X_val_scaled)

    classifier.fit(X_train_pca, y_train)

    y_pred = classifier.predict(X_val_pca)
    if hasattr(classifier, 'predict_proba'):
      y_prob = classifier.predict_proba(X_val_pca)[:, 1]
    elif hasattr(classifier, 'decision_function'):
      y_prob = classifier.decision_function(X_val_pca)
    else:
      y_prob = y_pred

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    sens = recall_score(y_val, y_pred, pos_label=1)
    spec = recall_score(y_val, y_pred, pos_label=0)
    prec = precision_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    metrics['acc'].append(acc)
    metrics['auc'].append(auc)
    metrics['sens'].append(sens)
    metrics['spec'].append(spec)
    metrics['prec'].append(prec)
    metrics['f1'].append(f1)
    metrics['n_components'].append(pca.n_components_)

  print(f'\n================= {model_name} Results =================')
  print(f'Mean Retained PCA Components: {np.mean(metrics["n_components"]):.1f}')
  print(
      f'Mean Accuracy:    {np.mean(metrics["acc"])*100:.2f}% (±'
      f' {np.std(metrics["acc"])*100:.2f}%)'
  )
  print(
      f'Mean AUC:         {np.mean(metrics["auc"]):.3f} (±'
      f' {np.std(metrics["auc"]):.3f})'
  )
  print(
      f'Mean Sensitivity: {np.mean(metrics["sens"]):.3f} (±'
      f' {np.std(metrics["sens"]):.3f})'
  )
  print(
      f'Mean Specificity: {np.mean(metrics["spec"]):.3f} (±'
      f' {np.std(metrics["spec"]):.3f})'
  )
  print(
      f'Mean F1-Score:    {np.mean(metrics["f1"]):.3f} (±'
      f' {np.std(metrics["f1"]):.3f})'
  )
  return metrics

In [16]:
# 6. Run Baselines
log_reg = LogisticRegression(penalty='l2', C=1.0, random_state=42, max_iter=1000)
svm_lin = SVC(kernel='linear', C=1.0, probability=True, random_state=42)

metrics_lr = evaluate_baseline_model(
    'Logistic Regression Baseline', log_reg, X, y
)
metrics_svm = evaluate_baseline_model('Linear SVM Baseline', svm_lin, X, y)


================= Logistic Regression Baseline Results =================
Mean Retained PCA Components: 201.6
Mean Accuracy:    97.09% (± 0.59%)
Mean AUC:         0.989 (± 0.011)
Mean Sensitivity: 0.962 (± 0.017)
Mean Specificity: 0.987 (± 0.016)
Mean F1-Score:    0.977 (± 0.005)

================= Linear SVM Baseline Results =================
Mean Retained PCA Components: 201.6
Mean Accuracy:    97.09% (± 0.96%)
Mean AUC:         0.993 (± 0.008)
Mean Sensitivity: 0.970 (± 0.023)
Mean Specificity: 0.973 (± 0.025)
Mean F1-Score:    0.977 (± 0.008)
